## Классификация: превышает ли значение CC50 медианное значение выборки

In [1]:
import pandas as pd

import sys
sys.path.append('../src')
from preprocessing import DatasetPreprocessor
from metrics_calculator import MetricsCalculator

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [3]:
df = pd.read_excel('../data/raw/classicMLCourseWorkData.xlsx')

# удаляем пропуски
clean_df = df.dropna().copy()
# медиана таргета
median_cc50 = clean_df['CC50, mM'].median()

# бинарный таргет
y = (clean_df['CC50, mM'] > median_cc50).astype(int)
X = clean_df.drop(columns=['CC50, mM'])

In [5]:
# Класс расчета метрик
metrics_helper = MetricsCalculator()

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#### Логистическая регрессия

In [6]:
# pipeline
logreg_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column=None,
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    (
        'model',
        LogisticRegression(max_iter=1000)
    )
])

# обучение
logreg_pipeline.fit(X_train, y_train)

# предсказания
logreg_y_train_pred = logreg_pipeline.predict(X_train)
logreg_y_test_pred = logreg_pipeline.predict(X_test)

# метрики
logreg_train_metrics = metrics_helper.classification_metrics(y_train, logreg_y_train_pred)
logreg_test_metrics = metrics_helper.classification_metrics(y_test, logreg_y_test_pred)

print('Метрики на train:')
display(logreg_train_metrics)

print('Метрики на test:')
display(logreg_test_metrics)

Метрики на train:


{'accuracy': 0.8847117794486216,
 'precision': 0.8925831202046036,
 'recall': 0.87468671679198,
 'f1': 0.8835443037974684}

Метрики на test:


{'accuracy': 0.8,
 'precision': 0.7941176470588235,
 'recall': 0.81,
 'f1': 0.801980198019802}

- Модель показывает хорошее качество на train (accuracy $\approx 0.88$), но хуже на test (accuracy $\approx 0.80$)
- Наблюдается некоторая просадка качества между train и test
- Метрики precision, recall и f1-score на test находятся на среднем уровне

**Вывод:** логистическая регрессия даёт неплохое базовое качество, но стоит протестировать более сложную модель

#### Случайный лес

In [7]:
rf_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column=None,
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    (
        'model',
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    )
])

# обучение
rf_pipeline.fit(X_train, y_train)

# предсказания
rf_y_train_pred = rf_pipeline.predict(X_train)
rf_y_test_pred = rf_pipeline.predict(X_test)

# метрики
rf_train_metrics = metrics_helper.classification_metrics(y_train, rf_y_train_pred)
rf_test_metrics = metrics_helper.classification_metrics(y_test, rf_y_test_pred)

print('Метрики на train:')
display(rf_train_metrics)

print('Метрики на test:')
display(rf_test_metrics)

Метрики на train:


{'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}

Метрики на test:


{'accuracy': 0.87,
 'precision': 0.8775510204081632,
 'recall': 0.86,
 'f1': 0.8686868686868687}

- Модель показывает идеальное качество на train (accuracy = 1.0), но хуже на test (accuracy $\approx 0.87$)
- Наблюдается переобучение
- По качеству значительно превосходит логистическую регрессию

**Вывод:** RandomForest даёт существенный прирост качества, требуется подбор гиперпараметров

In [8]:
# сетка гиперпараметров
rf_param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 10, None],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2],
    'model__max_features': ['sqrt', 0.5]
}

# GridSearch
rf_grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

# обучение
rf_grid_search.fit(X_train, y_train)

# лучшая модель
best_rf_pipeline = rf_grid_search.best_estimator_

print('Лучшие параметры:')
print(rf_grid_search.best_params_)

# предсказания
best_rf_y_train_pred = best_rf_pipeline.predict(X_train)
best_rf_y_test_pred = best_rf_pipeline.predict(X_test)

# метрики
best_rf_train_metrics = metrics_helper.classification_metrics(y_train, best_rf_y_train_pred)
best_rf_test_metrics = metrics_helper.classification_metrics(y_test, best_rf_y_test_pred)

print('Метрики на train:')
display(best_rf_train_metrics)

print('Метрики на test:')
display(best_rf_test_metrics)

Лучшие параметры:
{'model__max_depth': None, 'model__max_features': 0.5, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 200}
Метрики на train:


{'accuracy': 0.9987468671679198,
 'precision': 1.0,
 'recall': 0.9974937343358395,
 'f1': 0.998745294855709}

Метрики на test:


{'accuracy': 0.94,
 'precision': 0.9313725490196079,
 'recall': 0.95,
 'f1': 0.9405940594059405}

- Модель показывает почти идеальное качество на train (accuracy $\approx 0.99$) и высокое качество на test (accuracy $\approx 0.94$)
- Наблюдается переобучение, но значительно меньше, чем до подбора гиперпараметров
- Качество заметно улучшилось по сравнению с базовой версией RandomForest

In [10]:
# вероятности
best_rf_y_test_proba = best_rf_pipeline.predict_proba(X_test)[:, 1]

# полный набор метрик
best_rf_full_test_metrics = metrics_helper.full_classification_metrics(
    y_true=y_test,
    y_pred=best_rf_y_test_pred,
    y_proba=best_rf_y_test_proba
)

print('Полный набор метрик на test:')
display(best_rf_full_test_metrics)

Полный набор метрик на test:


{'accuracy': 0.94,
 'precision': 0.9313725490196079,
 'recall': 0.95,
 'f1': 0.9405940594059405,
 'roc_auc': 0.9740999999999999,
 'tn': np.int64(93),
 'fp': np.int64(7),
 'fn': np.int64(5),
 'tp': np.int64(95)}

- Модель показывает высокое качество на test (accuracy $\approx 0.94$, f1 $\approx 0.94$)
- ROC-AUC $\approx 0.97$, модель хорошо разделяет классы
- Ошибки невелики: FP = 7, FN = 5

**Вывод:** RandomForest демонстрирует высокое качество и является финальной моделью для задачи классификации CC50